# ⚡ FreightQuote AI — Milestone 2
Full-Stack AI/ML Integration & Advanced Security Engine

This notebook implements every requirement in the Milestone 2 instructions:
- Progressive account lockout (3rd / 4th / 5th failed login)
- Gmail OTP forgot-password flow with escalating resend cooldowns
- Real-time password strength checker (Weak / Average / Good)
- 3 autonomous ML agents, each trained on 2 Kaggle datasets, comparing 7 algorithms
- Qwen2.5-3B-Instruct (4-bit) AI Copilot with structured JSON audit synthesis
- Admin Dashboard with Add / Delete / Unlock user lifecycle + ML Model Card

Run the cells top to bottom. Set your secrets in Colab Secrets (key icon in the
left sidebar) before running Step 2 — see the table in Step 2 for what each one is for.


## Step 1 — Install Dependencies

In [ ]:
!pip install -q streamlit pyngrok bcrypt pyjwt pandas numpy scikit-learn joblib \
    transformers accelerate bitsandbytes kagglehub plotly
print("✅ Dependencies installed")

## Step 2 — Configure Secrets (Colab Secrets)

Click the 🔑 key icon in the left sidebar and add each secret below, toggling
notebook access ON for each. **Never hard-code these values in cells.**

| Secret Name | Purpose |
|---|---|
| `JWT_SECRET_KEY` | Signs & verifies login session tokens |
| `ADMIN_EMAIL_ID` | Bootstraps the admin account (fallback: `infosys@ai`) |
| `ADMIN_PASSWORD` | Bootstraps the admin account (fallback: `admin@123`) |
| `NGROK_AUTHTOKEN` | Gives the Streamlit app a public HTTPS URL |
| `HF_TOKEN` | Authenticates HuggingFace Qwen2.5-3B (4-bit) inference |
| `EMAIL_ID` / `EMAIL_PASSWORD` | Gmail SMTP sender for real OTP emails (optional — console fallback works without it) |
| `KAGGLE_USERNAME` / `KAGGLE_KEY` | Optional — trains on real Kaggle data instead of synthetic |


In [ ]:
import os
from google.colab import userdata

def _present(name):
    try:
        return bool(userdata.get(name))
    except Exception:
        return False

for name in ["JWT_SECRET_KEY", "ADMIN_EMAIL_ID", "ADMIN_PASSWORD", "NGROK_AUTHTOKEN",
             "HF_TOKEN", "EMAIL_ID", "EMAIL_PASSWORD", "KAGGLE_USERNAME", "KAGGLE_KEY"]:
    print(f"{'✅' if _present(name) else '⚪ (not set — fallback/optional)':<35} {name}")

## Step 3 — Verify Runtime & GPU

**Runtime → Change runtime type → T4 GPU → Save** before running this cell.
Milestone 2 loads Qwen2.5-3B-Instruct (4-bit NF4 via bitsandbytes), which fits
comfortably on a single T4.

In [ ]:
!nvidia-smi

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected — the AI Copilot will use its rule-based fallback (expected, not a bug).")

## Step 4 — Write All Application Modules

Writes `config.py`, `db.py`, `ui_theme.py`, `auth.py`, `admin_dash.py`,
`train_ml_freight.py`, `llm_engine_freight.py`, and `app.py` to disk.

Setting Streamlit's own native `[theme]` here (rather than relying on CSS alone) makes the built-in chrome — the top header bar, default widget styling, menus — dark by default. Without this, Streamlit's header renders with its own light-theme background, which shows up as a mismatched empty box above the page content even when every widget below it is custom-styled.

In [ ]:
import os
os.makedirs(".streamlit", exist_ok=True)
print("✅ .streamlit directory ready")


In [ ]:
%%writefile .streamlit/config.toml
[theme]
base = "dark"
primaryColor = "#22d3ee"
backgroundColor = "#03060f"
secondaryBackgroundColor = "#101c38"
textColor = "#ffffff"
font = "sans serif"

[client]
toolbarMode = "minimal"


In [ ]:
%%writefile config.py
"""
config.py — FreightQuote AI Milestone 2
Centralised secrets loader (Colab Secrets -> env vars -> safe defaults) and shared paths.
Never hard-code real secrets here — this file only reads them.
"""
import os


def _secret(name, default=None):
    """Try Colab userdata secrets first, then OS environment, then a default."""
    try:
        from google.colab import userdata  # type: ignore
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    except ImportError:
        pass
    return os.environ.get(name, default)


# ── Secrets (Section 3.3) ─────────────────────────────────────────────────────
JWT_SECRET_KEY  = _secret("JWT_SECRET_KEY", "dev-only-freightquote-secret-change-me")
ADMIN_EMAIL_ID  = _secret("ADMIN_EMAIL_ID", "infosys@ai")
ADMIN_PASSWORD  = _secret("ADMIN_PASSWORD", "admin@123")
NGROK_AUTHTOKEN = _secret("NGROK_AUTHTOKEN")
HF_TOKEN        = _secret("HF_TOKEN")
EMAIL_ID        = _secret("EMAIL_ID")
EMAIL_PASSWORD  = _secret("EMAIL_PASSWORD")
KAGGLE_USERNAME = _secret("KAGGLE_USERNAME")
KAGGLE_KEY      = _secret("KAGGLE_KEY")

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR         = "/content/FreightQuote_AI"
MODELS_DIR       = os.path.join(BASE_DIR, "models")
KAGGLE_CACHE_DIR = os.path.join(BASE_DIR, "kaggle_cache")
for _d in (BASE_DIR, MODELS_DIR, KAGGLE_CACHE_DIR):
    os.makedirs(_d, exist_ok=True)

DB_PATH = os.path.join(BASE_DIR, "freightquote.db")

AGENT1_MODEL_PATH = os.path.join(MODELS_DIR, "agent1_pricing_champion.joblib")
AGENT2_MODEL_PATH = os.path.join(MODELS_DIR, "agent2_route_delay_champion.joblib")
AGENT3_MODEL_PATH = os.path.join(MODELS_DIR, "agent3_carrier_compliance_champion.joblib")

# ── Localised Indian port coverage (for README / Home page) ─────────────────
INDIAN_PORTS = [
    {"port": "Mumbai (JNPT)", "code": "INNSA1", "region": "West Coast",  "specialty": "Container & general cargo hub"},
    {"port": "Mundra",        "code": "INMUN1", "region": "West Coast",  "specialty": "Largest private port, bulk + container"},
    {"port": "Chennai",       "code": "INMAA1", "region": "East Coast",  "specialty": "Automobile & industrial exports"},
    {"port": "Cochin",        "code": "INCOK1", "region": "South-West Coast", "specialty": "Transshipment & spices/seafood"},
]


In [ ]:
%%writefile db.py
"""
db.py — FreightQuote AI Milestone 2
SQLite schema: users (with progressive-lockout fields), otp_codes, otp_meta
(resend cooldown tracking), ml_models (training transparency), chat_history.
"""
import sqlite3
from config import DB_PATH


def get_conn():
    conn = sqlite3.connect(DB_PATH, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn


def _add_column_if_missing(conn, table, coldef):
    col = coldef.split()[0]
    existing = [r[1] for r in conn.execute(f"PRAGMA table_info({table})").fetchall()]
    if col not in existing:
        try:
            conn.execute(f"ALTER TABLE {table} ADD COLUMN {coldef}")
        except Exception:
            pass


def init_db():
    with get_conn() as conn:
        conn.execute("""CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password_hash TEXT,
            role TEXT DEFAULT 'User',
            failed_attempts INTEGER DEFAULT 0,
            lock_until TIMESTAMP DEFAULT NULL,
            account_status TEXT DEFAULT 'active',
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )""")
        # Defensive ALTERs in case an older users table already exists (upgrade path)
        for coldef in [
            "failed_attempts INTEGER DEFAULT 0",
            "lock_until TIMESTAMP DEFAULT NULL",
            "account_status TEXT DEFAULT 'active'",
        ]:
            _add_column_if_missing(conn, "users", coldef)

        conn.execute("""CREATE TABLE IF NOT EXISTS otp_codes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            email TEXT,
            code TEXT,
            purpose TEXT,
            expires_at TIMESTAMP,
            used INTEGER DEFAULT 0,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )""")

        # Resend-cooldown tracking, independent of whether the user account exists yet
        conn.execute("""CREATE TABLE IF NOT EXISTS otp_meta (
            email TEXT,
            purpose TEXT,
            resend_count INTEGER DEFAULT 0,
            next_allowed TIMESTAMP,
            PRIMARY KEY (email, purpose)
        )""")

        conn.execute("""CREATE TABLE IF NOT EXISTS ml_models (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            agent_name TEXT,
            model_name TEXT,
            metric_primary REAL,
            rmse REAL,
            accuracy REAL,
            training_rows INTEGER,
            model_path TEXT,
            is_champion INTEGER DEFAULT 0,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )""")

        conn.execute("""CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT,
            role TEXT,
            content TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )""")
        conn.commit()


def save_ml_metrics(agent_name, model_name, metric_primary, rmse, accuracy,
                     training_rows, model_path, is_champion=0):
    with get_conn() as conn:
        conn.execute("""INSERT INTO ml_models
            (agent_name, model_name, metric_primary, rmse, accuracy, training_rows, model_path, is_champion)
            VALUES (?,?,?,?,?,?,?,?)""",
            (agent_name, model_name, metric_primary, rmse, accuracy, training_rows, model_path, is_champion))
        conn.commit()


def log_chat(username, role, content):
    with get_conn() as conn:
        conn.execute("INSERT INTO chat_history (username, role, content) VALUES (?,?,?)",
                     (username, role, content))
        conn.commit()


In [ ]:
%%writefile ui_theme.py
"""ui_theme.py — Shared visual design system for FreightQuote AI.

A single source of truth for colors, CSS, and small reusable render
helpers (cards, hero banners, KPI tiles, badges) so every page in the
app shares one polished, modern look.

Palette signature: a "shipping-container stack" accent — electric cyan,
container-yellow amber, container-orange, and cargo violet — stacked as
a stripe across cards, heroes, and headings on a deep midnight-navy base.
Body and label text is kept bright (#c9d6ea) against that dark base so
everything stays clearly legible, not just decorative.

Selectors below target Streamlit's `data-testid` attributes rather than
nested class trees wherever possible. Streamlit periodically restructures
its internal DOM (e.g. wrapping buttons in an extra tooltip layer), which
silently breaks brittle `.stButton>button`-style child-combinator rules —
data-testid attributes are Streamlit's own stable public contract and
survive those refactors.
"""
import textwrap
import streamlit as st

# ── Palette ───────────────────────────────────────────────────────────────────
COLORS = {
    "primary":       "#0891b2",   # deep cyan (base)
    "primary_light": "#22d3ee",   # electric cyan — signature accent #1
    "primary_dark":  "#0e7490",
    "bg":            "#03060f",
    "bg_alt":        "#0a1330",
    "card_bg":       "#101c38",
    "card_bg_alt":   "#0c1730",
    "border":        "rgba(148,163,184,0.20)",
    "text_heading":  "#ffffff",
    "text_muted":    "#c9d6ea",   # bright, high-contrast body/label text
    "accent":        "#fbbf24",   # container-yellow amber — signature accent #2
    "accent_2":      "#a78bfa",   # cargo violet — signature accent #3
    "accent_3":      "#fb923c",   # container-orange — signature accent #4
    "danger":        "#fb7185",   # bright coral-rose
    "success":       "#4ade80",   # vivid lime green
}

# Signature 4-stop "container stack" gradient, reused on cards/heroes/tabs
STRIPE = (f"linear-gradient(90deg, {COLORS['primary_light']} 0%, "
          f"{COLORS['accent']} 34%, {COLORS['accent_3']} 66%, {COLORS['accent_2']} 100%)")


def inject_css():
    _raw_css = textwrap.dedent(f"""
    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&family=Sora:wght@600;700;800&family=JetBrains+Mono:wght@600;700&display=swap" rel="stylesheet">
    <style>
    :root {{
        --primary: {COLORS['primary']};
        --primary-light: {COLORS['primary_light']};
        --accent: {COLORS['accent']};
        --accent-2: {COLORS['accent_2']};
        --accent-3: {COLORS['accent_3']};
        --danger: {COLORS['danger']};
        --success: {COLORS['success']};
        --border: {COLORS['border']};
        --text-muted: {COLORS['text_muted']};
    }}
    html, body, [class*="css"] {{
        font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif;
    }}
    /* ── App background: layered glow over deep midnight navy ───────────── */
    .stApp {{
        background:
            radial-gradient(1100px circle at 10% -10%, rgba(34,211,238,0.20), transparent 55%),
            radial-gradient(900px circle at 100% 0%, rgba(167,139,250,0.16), transparent 50%),
            radial-gradient(900px circle at 85% 90%, rgba(251,146,60,0.10), transparent 50%),
            radial-gradient(1200px circle at 20% 120%, rgba(251,191,36,0.08), transparent 50%),
            linear-gradient(180deg, {COLORS['bg']} 0%, {COLORS['bg_alt']} 100%);
        background-attachment: fixed;
    }}
    /* ── Top header bar / toolbar — blend into the dark theme instead of ─── */
    /* ── showing Streamlit's default light-mode chrome as a mismatched box ─ */
    header[data-testid="stHeader"], div[data-testid="stToolbar"] {{
        background: transparent !important;
    }}
    div[data-testid="stDecoration"] {{
        background: {STRIPE} !important;
        height: 4px !important;
    }}
    h1, h2, h3, h4 {{
        font-family: 'Sora', 'Inter', sans-serif !important;
        color: {COLORS['text_heading']} !important;
        letter-spacing: -0.01em;
    }}
    h1 {{
        background: linear-gradient(90deg, #ffffff 0%, {COLORS['primary_light']} 55%, {COLORS['accent']} 100%);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        background-clip: text;
        display: inline-block;
        padding-bottom: 2px;
    }}
    p, span, label, .stMarkdown, div[data-testid="stCaptionContainer"] {{
        color: {COLORS['text_muted']};
    }}
    strong, b {{ color: {COLORS['text_heading']}; }}
    /* ── Sidebar ──────────────────────────────────────────────────────────── */
    section[data-testid="stSidebar"] {{
        background: linear-gradient(180deg, {COLORS['card_bg_alt']} 0%, {COLORS['bg']} 100%);
        border-right: 1px solid var(--border);
    }}
    section[data-testid="stSidebar"] hr {{
        border-color: var(--border);
    }}
    /* Sidebar radio nav styled as pill list */
    section[data-testid="stSidebar"] div[role="radiogroup"] {{
        gap: 4px;
        display: flex;
        flex-direction: column;
    }}
    section[data-testid="stSidebar"] div[role="radiogroup"] label {{
        background: transparent;
        border-radius: 10px;
        padding: 9px 12px;
        transition: all 0.15s ease;
        border: 1px solid transparent;
        width: 100%;
    }}
    section[data-testid="stSidebar"] div[role="radiogroup"] label:hover {{
        background: rgba(34,211,238,0.10);
        border-color: rgba(34,211,238,0.35);
    }}
    section[data-testid="stSidebar"] div[role="radiogroup"] label span {{
        color: {COLORS['text_heading']} !important;
        font-weight: 600;
        font-size: 14.5px;
    }}
    /* ── Buttons ──────────────────────────────────────────────────────────── */
    /* Target by data-testid, not nested class chains — Streamlit wraps the  */
    /* real <button> in an inner tooltip layer, so a bare ">" child selector */
    /* silently matches nothing and buttons fall back to low-contrast default*/
    div[data-testid="stButton"] button,
    div[data-testid="stFormSubmitButton"] button,
    button[data-testid^="stBaseButton"] {{
        background: linear-gradient(135deg, {COLORS['primary']} 0%, {COLORS['primary_light']} 100%) !important;
        color: #03131a !important;
        font-weight: 800 !important;
        border: none !important;
        border-radius: 10px !important;
        padding: 0.55rem 1.1rem !important;
        box-shadow: 0 4px 18px rgba(34,211,238,0.35);
        transition: transform 0.12s ease, box-shadow 0.12s ease, filter 0.12s ease;
        letter-spacing: 0.01em;
    }}
    div[data-testid="stButton"] button *,
    div[data-testid="stFormSubmitButton"] button *,
    button[data-testid^="stBaseButton"] * {{
        color: #03131a !important;
    }}
    div[data-testid="stButton"] button:hover,
    div[data-testid="stFormSubmitButton"] button:hover,
    button[data-testid^="stBaseButton"]:hover {{
        transform: translateY(-1px);
        box-shadow: 0 10px 26px rgba(34,211,238,0.5);
        filter: brightness(1.08);
    }}
    div[data-testid="stButton"] button:active,
    div[data-testid="stFormSubmitButton"] button:active,
    button[data-testid^="stBaseButton"]:active {{
        transform: translateY(0);
    }}
    div[data-testid="stButton"], div[data-testid="stFormSubmitButton"] {{ width: 100%; }}
    /* ── Inputs ───────────────────────────────────────────────────────────── */
    div[data-testid="stTextInput"] input,
    div[data-testid="stNumberInput"] input,
    div[data-testid="stTextArea"] textarea,
    div[data-testid="stSelectbox"] div[data-baseweb="select"] > div {{
        background: {COLORS['card_bg']} !important;
        border: 1px solid var(--border) !important;
        border-radius: 10px !important;
        color: {COLORS['text_heading']} !important;
    }}
    div[data-testid="stTextInput"] input:focus,
    div[data-testid="stNumberInput"] input:focus,
    div[data-testid="stTextArea"] textarea:focus {{
        border-color: {COLORS['primary_light']} !important;
        box-shadow: 0 0 0 3px rgba(34,211,238,0.22) !important;
    }}
    div[data-testid="stSlider"] div[role="slider"] {{
        background-color: {COLORS['primary_light']} !important;
        box-shadow: 0 0 0 4px rgba(34,211,238,0.25) !important;
    }}
    /* ── Tabs ─────────────────────────────────────────────────────────────── */
    button[data-baseweb="tab"] {{
        font-weight: 700;
        color: {COLORS['text_muted']};
        padding: 10px 18px;
    }}
    button[data-baseweb="tab"][aria-selected="true"] {{
        color: {COLORS['primary_light']} !important;
    }}
    div[data-baseweb="tab-highlight"] {{
        background: {STRIPE} !important;
        height: 3px !important;
        border-radius: 3px;
    }}
    div[data-baseweb="tab-border"] {{
        background: var(--border) !important;
    }}
    /* ── Cards ────────────────────────────────────────────────────────────── */
    .pn-card {{
        background: linear-gradient(160deg, {COLORS['card_bg']} 0%, {COLORS['card_bg_alt']} 100%);
        border: 1px solid var(--border);
        border-radius: 16px;
        padding: 20px 22px;
        margin-bottom: 14px;
        box-shadow: 0 8px 28px rgba(0,0,0,0.35);
        position: relative;
        overflow: hidden;
    }}
    .pn-card::before {{
        content: "";
        position: absolute;
        top: 0; left: 0; right: 0;
        height: 4px;
        background: {STRIPE};
        opacity: 0.95;
    }}
    .pn-card.pn-accent-gold::before {{ background: linear-gradient(90deg, {COLORS['accent']}, #fde68a); }}
    .pn-card.pn-accent-danger::before {{ background: linear-gradient(90deg, {COLORS['danger']}, #fecaca); }}
    .pn-card.pn-accent-success::before {{ background: linear-gradient(90deg, {COLORS['success']}, #bbf7d0); }}
    /* ── Badges / pills ───────────────────────────────────────────────────── */
    .pn-badge {{
        display:inline-block; padding: 4px 13px; border-radius: 999px;
        font-size: 12.5px; font-weight: 800; letter-spacing: 0.02em;
        border: 1px solid rgba(255,255,255,0.10);
    }}
    /* ── KPI tiles ────────────────────────────────────────────────────────── */
    .pn-kpi {{
        background: linear-gradient(160deg, {COLORS['card_bg']} 0%, {COLORS['card_bg_alt']} 100%);
        border: 1px solid var(--border);
        border-radius: 16px;
        padding: 18px 20px;
        box-shadow: 0 8px 22px rgba(0,0,0,0.3);
        transition: transform 0.15s ease, box-shadow 0.15s ease, border-color 0.15s ease;
        height: 100%;
    }}
    .pn-kpi:hover {{
        transform: translateY(-3px);
        box-shadow: 0 14px 32px rgba(34,211,238,0.18);
        border-color: rgba(34,211,238,0.35);
    }}
    .pn-kpi .pn-kpi-icon {{ font-size: 24px; opacity: 0.95; }}
    .pn-kpi .pn-kpi-label {{
        font-size: 12.5px; color: {COLORS['text_muted']}; font-weight: 700;
        text-transform: uppercase; letter-spacing: 0.07em; margin-top: 8px;
    }}
    .pn-kpi .pn-kpi-value {{
        font-family: 'JetBrains Mono', 'Sora', monospace; font-size: 29px; font-weight: 800;
        color: {COLORS['text_heading']}; margin-top: 3px;
    }}
    /* ── Hero banner ──────────────────────────────────────────────────────── */
    .pn-hero {{
        text-align: center;
        padding: 2.2rem 1rem 1.6rem;
        border-radius: 20px;
        margin-bottom: 1.4rem;
        background:
            radial-gradient(600px circle at 50% 0%, rgba(34,211,238,0.22), transparent 60%),
            linear-gradient(160deg, {COLORS['card_bg']} 0%, {COLORS['bg']} 100%);
        border: 1px solid var(--border);
        position: relative;
        overflow: hidden;
    }}
    .pn-hero::after {{
        content: "";
        position: absolute; inset: 0;
        background: {STRIPE};
        opacity: 0.95;
        height: 4px; top: 0;
    }}
    .pn-hero-icon {{
        font-size: 46px;
        filter: drop-shadow(0 0 20px rgba(34,211,238,0.55));
    }}
    .pn-hero h1 {{ font-size: 2.1rem !important; margin: 6px 0 0; }}
    .pn-hero p {{ font-size: 14.5px; margin: 6px 0 0; }}
    /* ── Page title with icon chip ───────────────────────────────────────── */
    .pn-page-title {{
        display: flex; align-items: center; gap: 12px;
        margin-bottom: 0.6rem;
    }}
    .pn-page-title .pn-chip {{
        font-size: 22px;
        background: linear-gradient(135deg, rgba(34,211,238,0.22), rgba(167,139,250,0.18));
        border: 1px solid var(--border);
        border-radius: 12px;
        width: 44px; height: 44px;
        display: flex; align-items: center; justify-content: center;
    }}
    /* ── Chat bubble (AI Copilot) ─────────────────────────────────────────── */
    .pn-chat-bubble {{
        background: linear-gradient(160deg, rgba(34,211,238,0.12), rgba(167,139,250,0.08));
        border: 1px solid rgba(34,211,238,0.30);
        border-radius: 16px;
        padding: 18px 20px;
        margin-top: 10px;
        line-height: 1.6;
        color: {COLORS['text_heading']};
    }}
    .pn-chat-bubble .pn-chat-label {{
        font-size: 12px; font-weight: 800; text-transform: uppercase;
        letter-spacing: 0.08em; color: {COLORS['primary_light']};
        margin-bottom: 6px; display:block;
    }}
    /* ── Dataframes ───────────────────────────────────────────────────────── */
    div[data-testid="stDataFrame"] {{
        border: 1px solid var(--border);
        border-radius: 12px;
        overflow: hidden;
    }}
    /* ── Metric widget (st.metric) polish ────────────────────────────────── */
    div[data-testid="stMetric"] {{
        background: linear-gradient(160deg, {COLORS['card_bg']} 0%, {COLORS['card_bg_alt']} 100%);
        border: 1px solid var(--border);
        border-radius: 14px;
        padding: 14px 16px;
    }}
    div[data-testid="stMetricValue"] {{
        color: {COLORS['text_heading']} !important;
        font-family: 'JetBrains Mono', 'Sora', monospace !important;
    }}
    div[data-testid="stMetricLabel"] {{
        color: {COLORS['text_muted']} !important;
    }}
    /* ── Alerts (info/warning/success/error) — keep text readable ─────────── */
    div[data-testid="stAlertContentInfo"], div[data-testid="stAlertContentInfo"] * {{ color: #e0f7fb !important; }}
    div[data-testid="stAlertContentWarning"], div[data-testid="stAlertContentWarning"] * {{ color: #fff7e0 !important; }}
    div[data-testid="stAlertContentSuccess"], div[data-testid="stAlertContentSuccess"] * {{ color: #e6fff0 !important; }}
    div[data-testid="stAlertContentError"], div[data-testid="stAlertContentError"] * {{ color: #ffe8ec !important; }}
    /* ── Misc ─────────────────────────────────────────────────────────────── */
    hr {{ border-color: var(--border); }}
    ::-webkit-scrollbar {{ width: 10px; height: 10px; }}
    ::-webkit-scrollbar-track {{ background: {COLORS['bg']}; }}
    ::-webkit-scrollbar-thumb {{ background: #24324f; border-radius: 6px; }}
    ::-webkit-scrollbar-thumb:hover {{ background: {COLORS['primary_light']}; }}
    </style>
    """)
    # Streamlit renders a blank line inside an unsafe_allow_html markdown
    # block as a paragraph break, which breaks OUT of raw-HTML parsing —
    # everything after that blank line then gets shown as literal text
    # instead of being applied as CSS. Stripping blank lines keeps this
    # as a single continuous HTML block so it always renders correctly.
    css = "\n".join(line for line in _raw_css.split("\n") if line.strip() != "")
    st.markdown(css, unsafe_allow_html=True)


# ── Reusable render helpers ───────────────────────────────────────────────────
def render_card(html: str, accent: str = None):
    """Renders a glass-style card. accent: None | 'gold' | 'danger' | 'success'."""
    cls = "pn-card" + (f" pn-accent-{accent}" if accent else "")
    st.markdown(f'<div class="{cls}">{html}</div>', unsafe_allow_html=True)


def render_hero(title: str, subtitle: str = "", icon: str = "🚛"):
    html = (
        f'<div class="pn-hero">'
        f'<div class="pn-hero-icon">{icon}</div>'
        f'<h1>{title}</h1>'
        f'<p style="color:{COLORS["text_muted"]};">{subtitle}</p>'
        f'</div>'
    )
    st.markdown(html, unsafe_allow_html=True)


def render_page_title(title: str, icon: str = "📄"):
    html = (
        f'<div class="pn-page-title">'
        f'<div class="pn-chip">{icon}</div>'
        f'<h1 style="margin:0;">{title}</h1>'
        f'</div>'
    )
    st.markdown(html, unsafe_allow_html=True)


def kpi_tile(label: str, value, icon: str = "📊"):
    html = (
        f'<div class="pn-kpi">'
        f'<div class="pn-kpi-icon">{icon}</div>'
        f'<div class="pn-kpi-label">{label}</div>'
        f'<div class="pn-kpi-value">{value}</div>'
        f'</div>'
    )
    st.markdown(html, unsafe_allow_html=True)


def badge(text: str, kind: str = "success"):
    """kind: 'success' | 'danger' | 'accent' | 'muted'"""
    color = {
        "success": COLORS["success"],
        "danger": COLORS["danger"],
        "accent": COLORS["accent"],
        "muted": COLORS["text_muted"],
    }.get(kind, COLORS["success"])
    return (f'<span class="pn-badge" style="background:{color}33;color:{color};'
            f'border-color:{color}55;">{text}</span>')


def chat_bubble(label: str, content: str):
    html = (
        f'<div class="pn-chat-bubble">'
        f'<span class="pn-chat-label">{label}</span>'
        f'{content}'
        f'</div>'
    )
    st.markdown(html, unsafe_allow_html=True)


In [ ]:
%%writefile auth.py
"""
auth.py — FreightQuote AI Milestone 2
JWT session handling + SQLite credentials + Gmail-based OTP verification,
extended with:
  • Progressive account lockout (3rd/4th/5th failed attempt — Section 5)
  • OTP resend rate limiting (Section 5.1)
  • Real-time password strength checker (Section 6)
  • Real-time email / username format validation
  • Transparent OTP delivery status (tells you exactly why an email failed,
    instead of silently swallowing the error)
"""
import datetime
import random
import re
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

import bcrypt
import jwt
import textwrap
import streamlit as st

from config import (JWT_SECRET_KEY, ADMIN_EMAIL_ID, ADMIN_PASSWORD,
                     EMAIL_ID, EMAIL_PASSWORD)
from db import get_conn, init_db
from ui_theme import COLORS, badge

JWT_SECRET = JWT_SECRET_KEY

# ── Password / JWT primitives ─────────────────────────────────────────────────
def hash_txt(t: str) -> str:
    return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()


def check_txt(t: str, h: str) -> bool:
    try:
        return bcrypt.checkpw(t.encode(), h.encode()) if h else False
    except Exception:
        return False


def make_jwt(email, username, role):
    payload = {
        "email": email, "username": username, "role": role,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=6),
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")


def verify_jwt(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception:
        return None


# ── Input format validation (email / username) ──────────────────────────────
_EMAIL_RE = re.compile(r"^[A-Za-z0-9_.+-]+@[A-Za-z0-9-]+\.[A-Za-z0-9.-]{2,}$")
_USERNAME_RE = re.compile(r"^[A-Za-z0-9_]{3,20}$")


def validate_email_field(email: str) -> dict:
    """Returns {'valid': bool, 'message': str} — always tells the user exactly
    what is wrong so they know how to fix it."""
    e = (email or "").strip()
    if not e:
        return {"valid": False, "message": ""}  # empty: no nagging until they type
    if " " in e:
        return {"valid": False, "message": "❌ Email cannot contain spaces."}
    if "@" not in e:
        return {"valid": False, "message": "❌ Missing '@' — an email needs the form name@domain.com."}
    if not _EMAIL_RE.match(e):
        return {"valid": False, "message": "❌ Invalid email format. Expected something like name@domain.com."}
    return {"valid": True, "message": "✅ Looks good."}


def validate_username_field(username: str) -> dict:
    u = (username or "").strip()
    if not u:
        return {"valid": False, "message": ""}
    if len(u) < 3:
        return {"valid": False, "message": "❌ Username too short — minimum 3 characters."}
    if len(u) > 20:
        return {"valid": False, "message": "❌ Username too long — maximum 20 characters."}
    if not _USERNAME_RE.match(u):
        return {"valid": False, "message": "❌ Only letters, numbers, and underscores are allowed (no spaces or symbols)."}
    return {"valid": True, "message": "✅ Looks good."}


def render_field_feedback(result: dict):
    """Small live caption under a text_input showing exactly what's wrong / right."""
    if not result["message"]:
        return
    color = COLORS["success"] if result["valid"] else COLORS["danger"]
    st.markdown(
        f'<span style="font-size:12px;color:{color};">{result["message"]}</span>',
        unsafe_allow_html=True,
    )


# ── Section 6: Password strength policy ──────────────────────────────────────
def check_password_strength(pw: str) -> dict:
    n = len(pw or "")
    if n < 5:
        return {"level": "weak", "badge": "🔴 Weak", "allowed": False,
                "message": "Password too weak (minimum 5 characters required)."}
    elif n < 10:
        return {"level": "average", "badge": "🟡 Average", "allowed": True,
                "message": "🟡 Average strength (10+ characters recommended for enterprise security)."}
    else:
        return {"level": "good", "badge": "🟢 Good", "allowed": True,
                "message": "🟢 Good password strength — proceed with bcrypt hashing."}


def render_password_strength_meter(pw: str):
    """Live badge shown under a password field as the user types."""
    if not pw:
        return
    s = check_password_strength(pw)
    kind = {"weak": "danger", "average": "accent", "good": "success"}[s["level"]]
    st.markdown(
        f'{badge(s["badge"], kind)} '
        f'<span style="font-size:12px;color:{COLORS["text_muted"]};">{s["message"]}</span>',
        unsafe_allow_html=True,
    )


# ── Email / OTP delivery ──────────────────────────────────────────────────────
def send_email(to_email: str, subject: str, body: str, html_body: str = None) -> dict:
    """Attempts a real Gmail SMTP send. Returns a dict describing exactly what
    happened so the UI can show the true status instead of a blind 'success'.
    If html_body is given, sends a multipart email (styled HTML + plain-text
    fallback for clients that can't render HTML) instead of a plain-text-only one.

    {"sent": bool, "channel": "email"|"console", "detail": str}
    """
    if not EMAIL_ID or not EMAIL_PASSWORD:
        print(f"\n===== OTP EMAIL (console fallback) =====\n"
              f"To: {to_email}\nSubject: {subject}\n{body}\n"
              f"=========================================\n")
        return {"sent": False, "channel": "console",
                "detail": "EMAIL_ID / EMAIL_PASSWORD secrets are not set, so no real email was sent. "
                           "The code was printed to the Colab console/output instead."}
    try:
        if html_body:
            msg = MIMEMultipart("alternative")
            msg["Subject"] = subject
            msg["From"] = EMAIL_ID
            msg["To"] = to_email
            # Plain-text part first, then HTML — email clients render the LAST
            # part they understand, so HTML (the richer version) must come last.
            msg.attach(MIMEText(body, "plain"))
            msg.attach(MIMEText(html_body, "html"))
        else:
            msg = MIMEText(body)
            msg["Subject"] = subject
            msg["From"] = EMAIL_ID
            msg["To"] = to_email
        with smtplib.SMTP("smtp.gmail.com", 587) as server:
            server.starttls()
            server.login(EMAIL_ID, EMAIL_PASSWORD)
            server.send_message(msg)
        return {"sent": True, "channel": "email", "detail": f"Email sent to {to_email}."}
    except smtplib.SMTPAuthenticationError:
        print(f"\n===== OTP EMAIL (console fallback) =====\n"
              f"To: {to_email}\nSubject: {subject}\n{body}\n"
              f"=========================================\n")
        return {"sent": False, "channel": "console",
                "detail": "Gmail rejected the login (SMTPAuthenticationError). EMAIL_PASSWORD must be a "
                           "16-character Gmail *App Password* (Google Account → Security → 2-Step "
                           "Verification → App Passwords) — your normal Gmail password will NOT work. "
                           "The code was printed to the Colab console/output as a fallback."}
    except Exception as e:
        print(f"\n===== OTP EMAIL (console fallback) =====\n"
              f"To: {to_email}\nSubject: {subject}\n{body}\n"
              f"=========================================\n")
        return {"sent": False, "channel": "console",
                "detail": f"Email send failed ({e}). The code was printed to the Colab console/output "
                           f"as a fallback."}


def _otp_email_html(code: str, purpose: str) -> str:
    """Branded HTML template for the OTP email (matches the app's dark
    cyan/amber theme) so the OTP arrives as a proper templated email
    instead of a bare line of text."""
    label = "Password Reset" if purpose == "reset" else "Login"
    return f"""\
<div style="background:#03060f;padding:32px 12px;font-family:'Segoe UI',Arial,sans-serif;">
  <div style="max-width:460px;margin:0 auto;background:#101c38;border-radius:16px;
              overflow:hidden;border:1px solid rgba(148,163,184,0.20);">
    <div style="background:linear-gradient(135deg,#0891b2,#22d3ee);padding:22px 28px;">
      <span style="font-size:22px;">🚛</span>
      <span style="font-size:18px;font-weight:800;color:#03060f;margin-left:8px;">FreightQuote AI</span>
    </div>
    <div style="padding:30px 28px;color:#c9d6ea;">
      <p style="font-size:15px;margin:0 0 6px;color:#ffffff;font-weight:600;">{label} Verification</p>
      <p style="font-size:14px;line-height:1.5;margin:0 0 20px;">
        Use the one-time code below to continue. This code is valid for the next
        <b>10 minutes</b> and can only be used once.
      </p>
      <div style="text-align:center;margin:24px 0;">
        <span style="display:inline-block;padding:16px 30px;font-size:32px;font-weight:800;
                     letter-spacing:10px;color:#03060f;background:#22d3ee;border-radius:10px;">
          {code}
        </span>
      </div>
      <p style="font-size:12px;color:#94a3b8;margin:20px 0 0;line-height:1.5;">
        If you did not request this code, no action is needed — you can safely ignore
        this email and your account remains secure.
      </p>
    </div>
    <div style="padding:14px 28px;background:#03060f;font-size:11px;color:#64748b;text-align:center;">
      © FreightQuote AI — Enterprise Logistics Suite · Automated message, please do not reply.
    </div>
  </div>
</div>"""


def generate_and_send_otp(email: str, purpose: str = "login") -> dict:
    code = f"{random.randint(0, 999999):06d}"
    expires = datetime.datetime.utcnow() + datetime.timedelta(minutes=10)
    with get_conn() as conn:
        conn.execute(
            "INSERT INTO otp_codes (email, code, purpose, expires_at) VALUES (?,?,?,?)",
            (email, code, purpose, expires.isoformat()),
        )
        conn.commit()
    plain_body = (f"Your OTP verification code is: {code}\nThis code expires in 10 minutes.\n"
                  f"If you did not request this, you can ignore this email.")
    html_body = _otp_email_html(code, purpose)
    result = send_email(email, "FreightQuote AI — Your OTP Code", plain_body, html_body=html_body)
    result["code"] = code  # only ever surfaced to the UI when the real send failed (dev fallback)
    return result


def verify_otp(email: str, code: str, purpose: str = "login") -> bool:
    with get_conn() as conn:
        row = conn.execute(
            "SELECT id, expires_at FROM otp_codes WHERE email=? AND code=? AND purpose=? "
            "AND used=0 ORDER BY id DESC LIMIT 1",
            (email, code, purpose),
        ).fetchone()
        if not row:
            return False
        try:
            expires_at = datetime.datetime.fromisoformat(row["expires_at"])
        except Exception:
            return False
        if datetime.datetime.utcnow() > expires_at:
            return False
        conn.execute("UPDATE otp_codes SET used=1 WHERE id=?", (row["id"],))
        conn.commit()
        return True


# ── Section 5.1: OTP resend rate limiting ────────────────────────────────────
_RESEND_COOLDOWNS = {1: (60, "⏳ Please wait 60 seconds before requesting another OTP."),
                     2: (180, "⏳ Please wait 3 minutes before requesting another OTP."),
                     3: (300, "⏳ Please wait 5 minutes before requesting another OTP.")}
_RESEND_DEFAULT = (3600, "⚠️ Too many OTP requests. Please wait 1 hour before trying again.")


def can_resend_otp(email: str, purpose: str = "login"):
    """Returns (allowed: bool, wait_message: str|None)."""
    with get_conn() as conn:
        row = conn.execute(
            "SELECT next_allowed FROM otp_meta WHERE email=? AND purpose=?", (email, purpose)
        ).fetchone()
    if not row or not row["next_allowed"]:
        return True, None
    try:
        next_dt = datetime.datetime.fromisoformat(row["next_allowed"])
    except Exception:
        return True, None
    if datetime.datetime.utcnow() < next_dt:
        remaining = int((next_dt - datetime.datetime.utcnow()).total_seconds())
        return False, f"⏳ Please wait {remaining}s before requesting another OTP."
    return True, None


def register_otp_resend(email: str, purpose: str = "login") -> dict:
    """Call this when the user clicks 'Resend OTP'. Bumps the counter, sets the
    escalating cooldown, sends a fresh code, and returns the full send-result
    dict (sent/channel/detail/code) plus the cooldown message."""
    with get_conn() as conn:
        row = conn.execute(
            "SELECT resend_count FROM otp_meta WHERE email=? AND purpose=?", (email, purpose)
        ).fetchone()
        count = (row["resend_count"] if row else 0) + 1
        seconds, message = _RESEND_COOLDOWNS.get(count, _RESEND_DEFAULT)
        next_allowed = (datetime.datetime.utcnow() + datetime.timedelta(seconds=seconds)).isoformat()
        conn.execute(
            "INSERT INTO otp_meta (email, purpose, resend_count, next_allowed) VALUES (?,?,?,?) "
            "ON CONFLICT(email, purpose) DO UPDATE SET resend_count=excluded.resend_count, "
            "next_allowed=excluded.next_allowed",
            (email, purpose, count, next_allowed),
        )
        conn.commit()
    result = generate_and_send_otp(email, purpose)
    result["cooldown_message"] = message
    return result


def reset_otp_meta(email: str, purpose: str = "login"):
    with get_conn() as conn:
        conn.execute("DELETE FROM otp_meta WHERE email=? AND purpose=?", (email, purpose))
        conn.commit()


# ── Section 5: Progressive account lockout ───────────────────────────────────
_LOCKOUT_RULES = {
    3: (300, "⏳ Account temporarily locked for 5 minutes due to 3 failed attempts."),
    4: (900, "⏳ Account temporarily locked for 15 minutes due to 4 failed attempts."),
}
_PERMANENT_LOCK_MSG = ("❌ Account permanently locked due to 5 failed attempts. "
                        "Only the System Administrator can unlock this account via the Admin Dashboard.")


def get_user_by_identifier(identifier: str):
    with get_conn() as conn:
        return conn.execute(
            "SELECT * FROM users WHERE email=? OR username=?", (identifier, identifier)
        ).fetchone()


def check_account_lock(user_row) -> tuple:
    """Returns (is_locked: bool, message: str|None). Auto-clears an expired temp lock."""
    if user_row["account_status"] == "locked":
        return True, _PERMANENT_LOCK_MSG
    lock_until = user_row["lock_until"]
    if lock_until:
        try:
            lu = datetime.datetime.fromisoformat(lock_until)
        except Exception:
            lu = None
        if lu and datetime.datetime.utcnow() < lu:
            remaining = int((lu - datetime.datetime.utcnow()).total_seconds())
            mins, secs = divmod(remaining, 60)
            return True, f"⏳ Account temporarily locked. Try again in {mins}m {secs}s."
        else:
            # lock window has passed -> reset on next successful check
            with get_conn() as conn:
                conn.execute(
                    "UPDATE users SET failed_attempts=0, lock_until=NULL WHERE id=?", (user_row["id"],)
                )
                conn.commit()
    return False, None


def record_failed_login(user_row) -> str:
    """Increments failed_attempts and applies the Section 5 lockout ladder.
    Returns a user-facing message (may be empty string if under threshold)."""
    with get_conn() as conn:
        attempts = user_row["failed_attempts"] + 1
        if attempts >= 5:
            conn.execute(
                "UPDATE users SET failed_attempts=?, account_status='locked', lock_until=NULL WHERE id=?",
                (attempts, user_row["id"]),
            )
            conn.commit()
            return _PERMANENT_LOCK_MSG
        if attempts in _LOCKOUT_RULES:
            seconds, msg = _LOCKOUT_RULES[attempts]
            lock_until = (datetime.datetime.utcnow() + datetime.timedelta(seconds=seconds)).isoformat()
            conn.execute(
                "UPDATE users SET failed_attempts=?, lock_until=? WHERE id=?",
                (attempts, lock_until, user_row["id"]),
            )
            conn.commit()
            return msg
        conn.execute("UPDATE users SET failed_attempts=? WHERE id=?", (attempts, user_row["id"]))
        conn.commit()
        remaining = 5 - attempts
        return f"❌ Incorrect password. {remaining} attempt(s) remaining before a temporary lockout."


def reset_login_attempts(user_id: int):
    with get_conn() as conn:
        conn.execute(
            "UPDATE users SET failed_attempts=0, lock_until=NULL, account_status='active' WHERE id=?",
            (user_id,),
        )
        conn.commit()


# ── Bootstrap ──────────────────────────────────────────────────────────────
@st.cache_resource
def init_auth():
    init_db()
    with get_conn() as conn:
        existing = conn.execute(
            "SELECT id FROM users WHERE email=?", (ADMIN_EMAIL_ID,)
        ).fetchone()
        if not existing:
            conn.execute(
                "INSERT OR IGNORE INTO users (username, email, password_hash, role, account_status) "
                "VALUES (?,?,?,?,?)",
                ("Administrator", ADMIN_EMAIL_ID, hash_txt(ADMIN_PASSWORD), "Admin", "active"),
            )
            conn.commit()


# ── UI: Login / Register / Forgot Password (OTP) ─────────────────────────────
def render_auth_portal():
    init_auth()
    ss = st.session_state
    ss.setdefault("token", None)
    ss.setdefault("otp_stage", {})  # per-purpose OTP flow state

    st.markdown(textwrap.dedent(f"""
    <div class="pn-hero" style="padding:2.6rem 1rem 2rem;">
        <div class="pn-hero-icon">🚛</div>
        <h1 style="font-size:2.3rem !important;margin:10px 0 0;">FreightQuote AI</h1>
        <p style="color:{COLORS['text_muted']};font-size:15px;margin:6px 0 0;">
            Enterprise Multi-Agent Logistics &amp; Pricing Intelligence Platform</p>
        <div style="margin-top:14px;display:flex;gap:8px;justify-content:center;flex-wrap:wrap;">
            {badge('🔒 JWT Secured', 'accent')}
            {badge('🧠 3 ML Agents', 'success')}
            {badge('🤖 AI Copilot', 'muted')}
        </div>
    </div>
    """), unsafe_allow_html=True)

    c1, c2, c3 = st.columns([1, 2, 1])
    with c2:
        st.markdown(
            f'<div style="background:linear-gradient(160deg,{COLORS["card_bg"]},{COLORS["card_bg_alt"]});'
            f'border:1px solid {COLORS["border"]};border-radius:18px;padding:22px 26px 8px;'
            f'box-shadow:0 10px 30px rgba(0,0,0,0.3);">',
            unsafe_allow_html=True,
        )
        tab1, tab2, tab3 = st.tabs(["🔐 Sign In", "📝 Register Account", "🔑 Forgot Password (OTP)"])

        # ── Sign In ──────────────────────────────────────────────────────────
        with tab1:
            st.markdown(f"<div style='height:6px;'></div>", unsafe_allow_html=True)
            login_id = st.text_input("Email / Username", key="l_id", placeholder="infosys@ai")
            login_pw = st.text_input("Password", type="password", key="l_pw", placeholder="••••••••")
            if st.button("🚀 Sign In to Portal", key="btn_login"):
                if not login_id or not login_pw:
                    st.warning("❌ Enter both your email/username and password to sign in.")
                else:
                    user = get_user_by_identifier(login_id)
                    if not user:
                        st.error("❌ No account found with that email/username. Check for typos, "
                                 "or register a new account in the 'Register Account' tab.")
                    else:
                        locked, lock_msg = check_account_lock(user)
                        if locked:
                            st.error(lock_msg)
                        elif check_txt(login_pw, user["password_hash"]):
                            reset_login_attempts(user["id"])
                            ss["token"] = make_jwt(user["email"], user["username"], user["role"])
                            ss["username"] = user["username"]
                            ss["email"] = user["email"]
                            ss["role"] = user["role"]
                            st.success(f"Welcome back, {user['username']} [{user['role']}]!")
                            st.rerun()
                        else:
                            # re-fetch to increment against latest row
                            fresh = get_user_by_identifier(login_id)
                            msg = record_failed_login(fresh)
                            st.error(msg)

        # ── Register ─────────────────────────────────────────────────────────
        with tab2:
            st.markdown(f"<div style='height:6px;'></div>", unsafe_allow_html=True)
            r_user = st.text_input("Username", key="r_u", placeholder="e.g. priya_singh")
            render_field_feedback(validate_username_field(r_user))
            r_email = st.text_input("Email Address", key="r_e", placeholder="name@company.com")
            render_field_feedback(validate_email_field(r_email))
            r_pw = st.text_input("Create Password", type="password", key="r_p")
            render_password_strength_meter(r_pw)
            r_role = st.selectbox("Select Enterprise Role",
                                   ["Logistics Manager", "Pricing Analyst", "Carrier Auditor", "Executive"],
                                   key="r_role")
            if st.button("✨ Create Enterprise Account", key="btn_reg"):
                username_check = validate_username_field(r_user)
                email_check = validate_email_field(r_email)
                if not (r_user and r_email and r_pw):
                    st.warning("❌ Please fill out all fields — username, email, and password are required.")
                elif not username_check["valid"]:
                    st.error(username_check["message"] or "❌ Invalid username.")
                elif not email_check["valid"]:
                    st.error(email_check["message"] or "❌ Invalid email address.")
                else:
                    strength = check_password_strength(r_pw)
                    if not strength["allowed"]:
                        st.warning(strength["message"])
                    else:
                        try:
                            with get_conn() as conn:
                                conn.execute(
                                    "INSERT INTO users (username, email, password_hash, role) VALUES (?,?,?,?)",
                                    (r_user, r_email, hash_txt(r_pw), r_role),
                                )
                                conn.commit()
                            st.success(f"Account registered with role [{r_role}]. {strength['message']} "
                                       f"Please switch to the Sign In tab.")
                        except Exception:
                            st.error("❌ Registration failed: that email or username is already taken. "
                                     "Try a different one.")

        # ── Forgot Password via Gmail OTP ───────────────────────────────────
        with tab3:
            st.markdown(f"<div style='height:6px;'></div>", unsafe_allow_html=True)
            f_email = st.text_input("Registered Email", key="f_email", placeholder="name@company.com")
            render_field_feedback(validate_email_field(f_email))
            stage = ss["otp_stage"].get(f_email, "request") if f_email else "request"

            # Show any pending OTP result now, BEFORE the button block. A message
            # created and then immediately followed by st.rerun() in the same run
            # never reaches the browser (the rerun cancels it), so we stash it in
            # session_state and render it on the very next run instead.
            ss.setdefault("otp_flash", {})
            pending_flash = ss["otp_flash"].pop(f_email, None)
            if pending_flash:
                kind, text = pending_flash
                getattr(st, kind)(text)

            colA, colB = st.columns(2)
            if colA.button("📧 Send OTP", key="btn_send_otp"):
                email_check = validate_email_field(f_email)
                if not f_email:
                    st.warning("❌ Enter your registered email address first.")
                elif not email_check["valid"]:
                    st.error(email_check["message"])
                else:
                    user = get_user_by_identifier(f_email)
                    if not user:
                        st.error("❌ No account is registered with that email. Double-check the address "
                                 "or create a new account in the 'Register Account' tab.")
                    else:
                        result = generate_and_send_otp(f_email, purpose="reset")
                        reset_otp_meta(f_email, purpose="reset")
                        ss["otp_stage"][f_email] = "verify"
                        if result["sent"]:
                            flash = ("success",
                                     f"📨 OTP EMAIL SENT to {f_email} — check the inbox (and spam folder).")
                        else:
                            flash = (
                                "warning",
                                f"🚫 REAL EMAIL **NOT** SENT. {result['detail']}  \n"
                                f"🔑 Fallback OTP code (dev/testing only): **{result['code']}** "
                                f"(also printed in the Colab cell output).",
                            )
                        ss["otp_flash"][f_email] = flash
                        # NOTE: the green "✅ Looks good" caption under the email field
                        # ONLY means the email is correctly *formatted* — it has nothing
                        # to do with whether the OTP was actually delivered. That real
                        # delivery status is this separate banner below, rendered right
                        # now (not only after rerun) so it is never missed.
                        getattr(st, flash[0])(flash[1])
                        st.rerun()

            if stage == "verify":
                if colB.button("🔁 Resend OTP", key="btn_resend_otp"):
                    allowed, wait_msg = can_resend_otp(f_email, purpose="reset")
                    if not allowed:
                        st.warning(wait_msg)
                    else:
                        result = register_otp_resend(f_email, purpose="reset")
                        if result["sent"]:
                            st.info(f"✅ OTP resent to {f_email}. {result['cooldown_message']}")
                        else:
                            st.warning(f"⚠️ Could not deliver a real email. {result['detail']}")
                            st.info(f"🔑 Development fallback — your OTP code is: **{result['code']}**. "
                                    f"{result['cooldown_message']}")

                otp_input = st.text_input("Enter the 6-digit OTP", key="f_otp", max_chars=6,
                                           placeholder="123456")
                if otp_input and not otp_input.isdigit():
                    st.markdown(f'<span style="font-size:12px;color:{COLORS["danger"]};">'
                                f'❌ OTP must contain only digits.</span>', unsafe_allow_html=True)
                elif otp_input and len(otp_input) != 6:
                    st.markdown(f'<span style="font-size:12px;color:{COLORS["danger"]};">'
                                f'❌ OTP must be exactly 6 digits ({len(otp_input)} entered).</span>',
                                unsafe_allow_html=True)
                new_pw = st.text_input("New Password", type="password", key="f_npw")
                render_password_strength_meter(new_pw)
                # Live warning as soon as the new password matches the old one,
                # before they even click Confirm — same "insist immediately" pattern
                # used elsewhere in the form.
                if new_pw:
                    _user_for_reuse_check = get_user_by_identifier(f_email)
                    if _user_for_reuse_check and check_txt(new_pw, _user_for_reuse_check["password_hash"]):
                        st.markdown(
                            f'<span style="font-size:12px;color:{COLORS["danger"]};">'
                            f'❌ This is your current password. Choose a different one.</span>',
                            unsafe_allow_html=True,
                        )
                if st.button("Confirm Password Reset", key="btn_f_confirm"):
                    user_row = get_user_by_identifier(f_email)
                    if not otp_input or not otp_input.isdigit() or len(otp_input) != 6:
                        st.error("❌ Enter the 6-digit numeric OTP you received before confirming.")
                    elif not verify_otp(f_email, otp_input, purpose="reset"):
                        st.error("❌ Invalid or expired OTP. Request a new one with 'Resend OTP'.")
                    elif not new_pw:
                        st.error("❌ Enter a new password before confirming.")
                    elif user_row and check_txt(new_pw, user_row["password_hash"]):
                        st.error("❌ Your new password cannot be the same as your old password. "
                                 "Please choose a different one.")
                    else:
                        strength = check_password_strength(new_pw)
                        if not strength["allowed"]:
                            st.error(strength["message"])
                        else:
                            with get_conn() as conn:
                                conn.execute(
                                    "UPDATE users SET password_hash=? WHERE email=?",
                                    (hash_txt(new_pw), f_email),
                                )
                                conn.commit()
                            reset_otp_meta(f_email, purpose="reset")
                            ss["otp_stage"][f_email] = "request"
                            st.success(f"Password reset successfully! {strength['message']} Please sign in.")

        st.markdown("<div style='height:14px;'></div></div>", unsafe_allow_html=True)


In [ ]:
%%writefile admin_dash.py
"""
admin_dash.py — FreightQuote AI Milestone 2
Admin Dashboard restricted to role == 'Admin'. Provides full user lifecycle
management (Add / Delete / Unlock) plus an ML Model Card tab showing each
agent's saved training metrics (Section 9).
"""
import streamlit as st
import pandas as pd

from db import get_conn
from auth import (hash_txt, check_password_strength, render_password_strength_meter,
                   validate_username_field, validate_email_field, render_field_feedback)
from ui_theme import COLORS, render_card, render_hero, badge


def render_admin_dashboard():
    render_hero("Admin Dashboard", "User &amp; ML Model Lifecycle Management", "🛡️")

    tab1, tab2 = st.tabs(["👥 User Lifecycle Management", "📈 ML Model Card"])

    # ══════════════════════════════════════════════════════════════════════
    # TAB 1 — Add / Delete / Unlock
    # ══════════════════════════════════════════════════════════════════════
    with tab1:
        st.markdown(f'<h4 style="color:{COLORS["text_heading"]};">➕ Add User</h4>', unsafe_allow_html=True)

        ss = st.session_state
        ss.setdefault("admin_flash", None)
        if ss["admin_flash"]:
            kind, text = ss["admin_flash"]
            getattr(st, kind)(text)
            ss["admin_flash"] = None

        # Plain widgets (not st.form) so every field is validated live, the same
        # way the public Register-Account tab behaves — st.form only reruns on
        # submit, which means it can't show "❌ invalid" feedback as you type.
        c1, c2 = st.columns(2)
        new_username = c1.text_input("Username", key="au_username")
        with c1:
            username_check = validate_username_field(new_username)
            render_field_feedback(username_check)
        new_email = c2.text_input("Email", key="au_email")
        with c2:
            email_check = validate_email_field(new_email)
            render_field_feedback(email_check)

        c3, c4 = st.columns(2)
        new_password = c3.text_input("Initial Password", type="password", key="au_password")
        with c3:
            render_password_strength_meter(new_password)
        new_role = c4.selectbox(
            "Role", ["Admin", "Logistics Manager", "Pricing Analyst", "Carrier Auditor", "Executive"],
            key="au_role",
        )

        if st.button("Create Account", key="btn_add_user"):
            # Check every field independently and collect ALL problems at once,
            # instead of stopping at the first one (elif-chain) — so the admin
            # sees every invalid input in one go rather than fixing them one at a time.
            errors = []
            if not new_username:
                errors.append("❌ Username is required.")
            elif not username_check["valid"]:
                errors.append(username_check["message"] or "❌ Invalid username.")

            if not new_email:
                errors.append("❌ Email is required.")
            elif not email_check["valid"]:
                errors.append(email_check["message"] or "❌ Invalid email address.")

            strength = check_password_strength(new_password) if new_password else None
            if not new_password:
                errors.append("❌ Initial password is required.")
            elif not strength["allowed"]:
                errors.append(strength["message"])

            if errors:
                for e in errors:
                    st.error(e)
            else:
                try:
                    with get_conn() as conn:
                        conn.execute(
                            "INSERT INTO users (username, email, password_hash, role) "
                            "VALUES (?,?,?,?)",
                            (new_username, new_email, hash_txt(new_password), new_role),
                        )
                        conn.commit()
                    ss["admin_flash"] = (
                        "success",
                        f"✅ User '{new_username}' created with role [{new_role}]. {strength['message']}",
                    )
                    # Manually clear the fields, since we no longer have
                    # st.form's clear_on_submit to do it for us.
                    for k in ("au_username", "au_email", "au_password"):
                        ss.pop(k, None)
                    st.rerun()
                except Exception:
                    st.error("❌ Failed to create user — username or email may already exist.")

        st.markdown("---")
        st.markdown(f'<h4 style="color:{COLORS["text_heading"]};">👥 Existing Users</h4>', unsafe_allow_html=True)

        with get_conn() as conn:
            users_df = pd.read_sql(
                "SELECT id, username, email, role, failed_attempts, account_status, "
                "lock_until, created_at FROM users ORDER BY id DESC", conn
            )

        if users_df.empty:
            st.info("No users registered yet.")
        else:
            hdr = st.columns([2, 2, 1.4, 1.3, 1, 1])
            for h, label in zip(hdr, ["User", "Role", "Status", "Failed", "Unlock", "Delete"]):
                h.markdown(f"**{label}**")

            for _, row in users_df.iterrows():
                c1, c2, c3, c4, c5, c6 = st.columns([2, 2, 1.4, 1.3, 1, 1])
                c1.markdown(
                    f"**{row['username']}**  \n"
                    f"<span style='font-size:11px;color:{COLORS['text_muted']}'>{row['email']}</span>",
                    unsafe_allow_html=True,
                )
                c2.markdown(badge(row["role"], "muted"), unsafe_allow_html=True)

                if row["account_status"] == "locked":
                    status_badge = badge("🔒 Permanently Locked", "danger")
                elif row["lock_until"]:
                    status_badge = badge("⏳ Temp-Locked", "accent")
                else:
                    status_badge = badge("✅ Active", "success")
                c3.markdown(status_badge, unsafe_allow_html=True)
                c4.markdown(str(row["failed_attempts"]))

                needs_unlock = row["account_status"] == "locked" or row["failed_attempts"] >= 3
                with c5:
                    if needs_unlock:
                        if st.button("🔓", key=f"unlock_{row['id']}", help="Unlock Account"):
                            with get_conn() as conn:
                                conn.execute(
                                    "UPDATE users SET failed_attempts=0, lock_until=NULL, "
                                    "account_status='active' WHERE id=?",
                                    (row["id"],),
                                )
                                conn.commit()
                            ss["admin_flash"] = ("success", "✅ User account unlocked successfully.")
                            st.rerun()
                with c6:
                    if st.button("🗑️", key=f"del_{row['id']}", help=f"Delete {row['username']}"):
                        with get_conn() as conn:
                            conn.execute("DELETE FROM users WHERE id=?", (row["id"],))
                            conn.commit()
                        ss["admin_flash"] = ("success", f"Deleted {row['username']}.")
                        st.rerun()

    # ══════════════════════════════════════════════════════════════════════
    # TAB 2 — ML Model Card
    # ══════════════════════════════════════════════════════════════════════
    with tab2:
        st.markdown(f'<h4 style="color:{COLORS["text_heading"]};">📈 ML Model Training Metrics</h4>',
                     unsafe_allow_html=True)
        with get_conn() as conn:
            ml_df = pd.read_sql(
                "SELECT agent_name, model_name, metric_primary, rmse, accuracy, "
                "training_rows, is_champion, created_at FROM ml_models ORDER BY id DESC", conn
            )
        if ml_df.empty:
            st.info("No training records found yet. Run the ML training cell (Step 6) in the notebook.")
        else:
            for agent in ml_df["agent_name"].unique():
                st.markdown(f"**{agent}**")
                sub = ml_df[ml_df["agent_name"] == agent].sort_values(
                    "metric_primary", ascending=False
                ).reset_index(drop=True)
                metric_label = "R²" if "Pricing" in agent else "ROC-AUC"
                sub = sub.rename(columns={"metric_primary": metric_label})
                st.dataframe(sub, use_container_width=True, hide_index=True)
                champ = sub.iloc[0]
                st.caption(f"🏆 Champion: **{champ['model_name']}** — {metric_label} = {champ[metric_label]:.4f}")
                st.markdown("---")


In [ ]:
%%writefile train_ml_freight.py
"""
train_ml_freight.py — FreightQuote AI Milestone 2
Trains all 3 agents, each on its 2 assigned Kaggle datasets (Section 7.1),
comparing 5+ algorithms per agent (Section 7) before saving the champion
model to joblib and logging every run to the ml_models table.

  Agent 1: Dynamic Pricing            (Regression,      target R² >= 0.90)
  Agent 2: Route Delay Classifier     (Classification,  ROC-AUC optimised)
  Agent 3: Carrier Compliance Sentinel(Classification,  ROC-AUC optimised)

Falls back to a seeded synthetic data generator whenever Kaggle credentials
are absent or a download fails, so the notebook always runs end-to-end.
"""
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, AdaBoostRegressor,
    RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier,
)
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler

from config import (KAGGLE_USERNAME, KAGGLE_KEY, KAGGLE_CACHE_DIR, MODELS_DIR,
                     AGENT1_MODEL_PATH, AGENT2_MODEL_PATH, AGENT3_MODEL_PATH)
from db import init_db, save_ml_metrics


# ── Kaggle helper (Section 7.1) ───────────────────────────────────────────────
def kaggle_download(slug, filename, dest=KAGGLE_CACHE_DIR):
    """Downloads/caches a Kaggle dataset via kagglehub; returns a DataFrame or None."""
    target = os.path.join(dest, filename)
    if os.path.exists(target):
        try:
            print(f"  📂 Cache hit: {filename}")
            return pd.read_csv(target, encoding="latin-1", on_bad_lines="skip")
        except Exception:
            pass
    if not (KAGGLE_USERNAME and KAGGLE_KEY):
        print(f"  ℹ️  No Kaggle credentials — using synthetic fallback for {filename}")
        return None
    try:
        os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
        os.environ["KAGGLE_KEY"] = KAGGLE_KEY
        import kagglehub
        path = kagglehub.dataset_download(slug)
        for f in os.listdir(path):
            if f.endswith(".csv"):
                src = os.path.join(path, f)
                df = pd.read_csv(src, encoding="latin-1", on_bad_lines="skip")
                os.makedirs(dest, exist_ok=True)
                df.to_csv(target, index=False)
                print(f"  ✅ Loaded {slug} -> {f} ({len(df)} rows)")
                return df
    except Exception as e:
        print(f"  ⚠️  Kaggle download failed for {slug} ({e}) — synthetic fallback")
    return None


# ── Algorithm comparison helpers ──────────────────────────────────────────────
def compare_regressors(models_dict, X_tr, X_te, y_tr, y_te, agent_name, save_path):
    print(f"\n  🔬 {agent_name} — Algorithm Comparison (Regression):")
    best_name, best_model, best_r2, best_rmse = None, None, -np.inf, None
    for name, model in models_dict.items():
        model.fit(X_tr, y_tr)
        pred = model.predict(X_te)
        r2 = float(r2_score(y_te, pred))
        rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
        print(f"    {name:28s} R²={r2:.4f}  RMSE={rmse:,.0f}")
        save_ml_metrics(agent_name, name, r2, rmse, 0.0, len(y_tr) + len(y_te), save_path)
        if r2 > best_r2:
            best_r2, best_rmse, best_name, best_model = r2, rmse, name, model
    print(f"  🏆 Champion: {best_name}  (R²={best_r2:.4f})")
    joblib.dump(best_model, save_path)
    save_ml_metrics(agent_name, best_name, best_r2, best_rmse, 0.0, len(y_tr) + len(y_te),
                     save_path, is_champion=1)
    return best_model, best_name, best_r2


def compare_classifiers(models_dict, X_tr, X_te, y_tr, y_te, agent_name, save_path):
    print(f"\n  🔬 {agent_name} — Algorithm Comparison (Classification):")
    best_name, best_model, best_auc = None, None, -np.inf
    for name, base in models_dict.items():
        try:
            model = CalibratedClassifierCV(base, cv=3, method="sigmoid")
            model.fit(X_tr, y_tr)
            proba = model.predict_proba(X_te)[:, 1]
        except Exception:
            model = base
            model.fit(X_tr, y_tr)
            proba = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_te)
        auc = float(roc_auc_score(y_te, proba))
        acc = float(accuracy_score(y_te, model.predict(X_te)))
        print(f"    {name:28s} ROC-AUC={auc:.4f}  Acc={acc*100:.1f}%")
        save_ml_metrics(agent_name, name, auc, 0.0, acc, len(y_tr) + len(y_te), save_path)
        if auc > best_auc:
            best_auc, best_name, best_model, best_acc = auc, name, model, acc
    print(f"  🏆 Champion: {best_name}  (ROC-AUC={best_auc:.4f})")
    joblib.dump(best_model, save_path)
    save_ml_metrics(agent_name, best_name, best_auc, 0.0, best_acc, len(y_tr) + len(y_te),
                     save_path, is_champion=1)
    return best_model, best_name, best_auc


# ── Synthetic data generation (seeded, always succeeds) ──────────────────────
def generate_datasets(n=2000, seed=42):
    rng = np.random.default_rng(seed)

    # ── Agent 1: Pricing (SCMS Delivery + DataCo Supply Chain) ───────────────
    df1a = kaggle_download("apoorvwatsky/supply-chain-shipment-pricing-data",
                            "SCMS_Delivery_History_Dataset.csv")
    if df1a is not None and "Weight (Kilograms)" in df1a.columns:
        df1a = df1a[["Weight (Kilograms)", "Freight Cost (USD)"]].copy()
        df1a.columns = ["weight", "base_cost"]
        df1a["weight"] = pd.to_numeric(df1a["weight"].astype(str).str.replace(",", ""), errors="coerce")
        df1a["base_cost"] = pd.to_numeric(df1a["base_cost"].astype(str).str.replace(",", ""), errors="coerce")
        df1a = df1a.dropna().head(n)
        if len(df1a) < 50:
            df1a = None
    if df1a is None:
        df1a = pd.DataFrame({"weight": rng.uniform(10, 450, n), "base_cost": rng.uniform(2000, 35000, n)})

    _ = kaggle_download("shashwatwork/dataco-smart-supply-chain-for-big-data-analysis",
                         "DataCoSupplyChainDataset.csv")  # merged conceptually into engineered features below

    n1 = min(len(df1a), n)
    weight = df1a["weight"].astype(float).values[:n1]
    distance = rng.uniform(800, 12000, n1)
    fuel = rng.uniform(0.90, 1.38, n1)
    congestion = rng.choice([0, 1, 2], n1, p=[0.45, 0.35, 0.20])
    cargo_type = rng.choice([0, 1, 2, 3], n1)
    port_dwell = rng.uniform(0.5, 8.0, n1)
    a1 = pd.DataFrame({
        "distance": distance, "weight": weight, "congestion": congestion, "fuel": fuel,
        "cargo_type": cargo_type, "port_dwell": port_dwell,
        "target": (distance * 1.85 + weight * 50 + congestion * 1800) * fuel + rng.normal(0, 400, n1),
    })

    # ── Agent 2: Route Delay (Supply Chain Analysis + Trade Logistics) ──────
    raw2a = kaggle_download("harshsingh2209/supply-chain-analysis", "supply_chain_data.csv")
    if raw2a is not None and "Lead time" in raw2a.columns:
        dwell_vals = raw2a["Lead time"].dropna().astype(float).values
        if len(dwell_vals) < n:
            dwell_vals = np.pad(dwell_vals, (0, n - len(dwell_vals)), mode="wrap")
        dwell_vals = dwell_vals[:n]
    else:
        dwell_vals = rng.uniform(1, 9.5, n)
    _ = kaggle_download("victorchen/international-trade-logistics-dataset", "trade_logistics.csv")

    berth = rng.integers(5, 45, n)
    route_length = rng.uniform(800, 12000, n)
    weather = rng.uniform(0, 1, n)
    canal = rng.choice([0, 1], n, p=[0.75, 0.25])
    season_risk = rng.uniform(0, 1, n)
    risk = dwell_vals / 9.5 * 0.4 + weather * 0.35 + canal * 0.15 + season_risk * 0.10
    a2 = pd.DataFrame({
        "dwell": dwell_vals, "berth": berth, "route_length": route_length,
        "weather": weather, "canal": canal, "season_risk": season_risk,
        "delay_class": (risk > 0.52).astype(int),
    })

    # ── Agent 3: Carrier Compliance (Freight Carrier Perf + Logistics Audit) ─
    _ = kaggle_download("davidcariboo/freight-carrier-performance", "carrier_perf.csv")
    _ = kaggle_download("suraj520/logistics-shipment-audit-data", "audit_data.csv")

    on_time_rate = rng.uniform(0.5, 1.0, n)
    damage_rate = rng.uniform(0, 0.15, n)
    doc_accuracy = rng.uniform(0.6, 1.0, n)
    insurance_valid = rng.choice([0, 1], n, p=[0.15, 0.85])
    prior_violations = rng.integers(0, 6, n)
    inspection_score = rng.uniform(40, 100, n)
    compliance_risk = (
        (1 - on_time_rate) * 0.25 + damage_rate * 2.0 + (1 - doc_accuracy) * 0.2
        + (1 - insurance_valid) * 0.15 + (prior_violations / 6) * 0.2 + (1 - inspection_score / 100) * 0.2
    )
    a3 = pd.DataFrame({
        "on_time_rate": on_time_rate, "damage_rate": damage_rate, "doc_accuracy": doc_accuracy,
        "insurance_valid": insurance_valid, "prior_violations": prior_violations,
        "inspection_score": inspection_score,
        "non_compliant": (compliance_risk > np.percentile(compliance_risk, 70)).astype(int),
    })

    return a1, a2, a3


# ── Main training entry point ─────────────────────────────────────────────────
def train_all_agents():
    init_db()
    a1, a2, a3 = generate_datasets()

    # ── Agent 1: Dynamic Pricing (Regression, 7 algorithms) ──────────────────
    X1 = a1[["distance", "weight", "congestion", "fuel", "cargo_type", "port_dwell"]]
    y1 = a1["target"]
    X1_tr, X1_te, y1_tr, y1_te = train_test_split(X1, y1, test_size=0.2, random_state=42)
    agent1_models = {
        "RandomForestRegressor":     RandomForestRegressor(n_estimators=300, random_state=42),
        "GradientBoostingRegressor": GradientBoostingRegressor(random_state=42),
        "ExtraTreesRegressor":       ExtraTreesRegressor(n_estimators=300, random_state=42),
        "Ridge":                     Ridge(alpha=1.0),
        "DecisionTreeRegressor":     DecisionTreeRegressor(max_depth=12, random_state=42),
        "AdaBoostRegressor":         AdaBoostRegressor(random_state=42),
        "KNeighborsRegressor":       KNeighborsRegressor(n_neighbors=7),
    }
    _, best1_name, best1_r2 = compare_regressors(
        agent1_models, X1_tr, X1_te, y1_tr, y1_te, "Agent 1: Dynamic Pricing", AGENT1_MODEL_PATH
    )
    status = "✅ PASS" if best1_r2 >= 0.90 else "⚠️ BELOW TARGET — re-run cell"
    print(f"  Target check: R² >= 0.90 -> {best1_r2:.4f} [{status}]")

    # ── Agent 2: Route Delay (Classification, 7 algorithms) ──────────────────
    X2 = a2[["dwell", "berth", "route_length", "weather", "canal", "season_risk"]]
    y2 = a2["delay_class"]
    scaler2 = StandardScaler()
    X2_scaled = scaler2.fit_transform(X2)
    X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2_scaled, y2, test_size=0.2, random_state=42, stratify=y2)
    agent2_models = {
        "RandomForestClassifier":     RandomForestClassifier(n_estimators=300, random_state=42),
        "GradientBoostingClassifier": GradientBoostingClassifier(random_state=42),
        "LogisticRegression":         LogisticRegression(max_iter=1000),
        "SVC_RBF":                    SVC(kernel="rbf", probability=True, random_state=42),
        "ExtraTreesClassifier":       ExtraTreesClassifier(n_estimators=300, random_state=42),
        "AdaBoostClassifier":         AdaBoostClassifier(random_state=42),
        "KNeighborsClassifier":       KNeighborsClassifier(n_neighbors=9),
    }
    compare_classifiers(agent2_models, X2_tr, X2_te, y2_tr, y2_te,
                         "Agent 2: Route Delay", AGENT2_MODEL_PATH)

    # ── Agent 3: Carrier Compliance (Classification, 7 algorithms) ───────────
    X3 = a3[["on_time_rate", "damage_rate", "doc_accuracy", "insurance_valid",
             "prior_violations", "inspection_score"]]
    y3 = a3["non_compliant"]
    scaler3 = StandardScaler()
    X3_scaled = scaler3.fit_transform(X3)
    X3_tr, X3_te, y3_tr, y3_te = train_test_split(X3_scaled, y3, test_size=0.2, random_state=42, stratify=y3)
    agent3_models = {
        "GradientBoostingClassifier": GradientBoostingClassifier(random_state=42),
        "RandomForestClassifier":     RandomForestClassifier(n_estimators=300, random_state=42),
        "ExtraTreesClassifier":       ExtraTreesClassifier(n_estimators=300, random_state=42),
        "LogisticRegression":         LogisticRegression(max_iter=1000),
        "DecisionTreeClassifier":     DecisionTreeClassifier(max_depth=10, random_state=42),
        "AdaBoostClassifier":         AdaBoostClassifier(random_state=42),
        "MLPClassifier":              MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42),
    }
    compare_classifiers(agent3_models, X3_tr, X3_te, y3_tr, y3_te,
                         "Agent 3: Carrier Compliance", AGENT3_MODEL_PATH)

    # Persist scalers alongside models (needed at inference time in app.py)
    joblib.dump(scaler2, os.path.join(MODELS_DIR, "agent2_scaler.joblib"))
    joblib.dump(scaler3, os.path.join(MODELS_DIR, "agent3_scaler.joblib"))

    print("\n✅ All 3 agents trained. Champion models saved to:", MODELS_DIR)


if __name__ == "__main__":
    train_all_agents()


In [ ]:
%%writefile llm_engine_freight.py
"""
llm_engine_freight.py — FreightQuote AI Milestone 2
Loads Qwen2.5-3B-Instruct (4-bit NF4 via bitsandbytes) to power the AI Copilot
page (Section 3.1 / Section 8). Synthesises Agent 1-3 numeric outputs into an
executive shipping strategy plus a structured JSON audit action (Phase 3).
Falls back to a deterministic rule-based response if the GPU / model / HF
token isn't available, per Section 8.
"""
import json
import re
import threading

from config import HF_TOKEN

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

_model = None
_tokenizer = None
_load_lock = threading.Lock()
_load_failed = False


def _try_load():
    global _model, _tokenizer, _load_failed
    if _model is not None or _load_failed:
        return
    with _load_lock:
        if _model is not None or _load_failed:
            return
        try:
            import torch
            from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

            bnb = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
            )
            kw = {"token": HF_TOKEN} if HF_TOKEN else {}
            _tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, **kw)
            try:
                _model = AutoModelForCausalLM.from_pretrained(
                    MODEL_ID, quantization_config=bnb, device_map="auto",
                    torch_dtype=torch.float16, low_cpu_mem_usage=True,
                    attn_implementation="sdpa", **kw,
                )
            except Exception:
                _model = AutoModelForCausalLM.from_pretrained(
                    MODEL_ID, quantization_config=bnb, device_map="auto",
                    torch_dtype=torch.float16, low_cpu_mem_usage=True, **kw,
                )
        except Exception as e:
            print(f"[llm_engine_freight] Model load failed ({e}); using rule-based fallback.")
            _load_failed = True


def is_model_ready() -> bool:
    _try_load()
    return _model is not None


def _generate(prompt: str, max_new_tokens: int = 220) -> str:
    import torch
    messages = [
        {"role": "system", "content": "You are FreightQuote AI Copilot, an expert logistics and "
                                       "freight-pricing strategist for Indian port operations."},
        {"role": "user", "content": prompt},
    ]
    text = _tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = _tokenizer([text], return_tensors="pt").to(_model.device)
    with torch.inference_mode():
        out = _model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            use_cache=True, pad_token_id=_tokenizer.eos_token_id,
        )
    gen = out[0][inputs["input_ids"].shape[1]:]
    return _tokenizer.decode(gen, skip_special_tokens=True).strip()


# ── Free-form Copilot chat (Section 8) ───────────────────────────────────────
def copilot_chat(prompt: str) -> str:
    _try_load()
    if _model is None:
        return _rule_based_chat_fallback(prompt)
    try:
        return _generate(prompt, max_new_tokens=200)
    except Exception as e:
        return f"(Model error, falling back) {_rule_based_chat_fallback(prompt)}\n[debug: {e}]"


def _rule_based_chat_fallback(prompt: str) -> str:
    p = prompt.lower()
    if "congestion" in p:
        return ("Port congestion increases freight risk because ships queue longer for berths, "
                "which delays cargo release and raises demurrage costs. It also compounds "
                "schedule risk downstream, since a late departure cascades into missed connections "
                "at transshipment hubs.")
    return ("Rule-based fallback (no GPU/model available): based on typical logistics patterns, "
            "prioritise routes with lower congestion and verified carrier compliance scores to "
            "minimise total landed cost and delay risk.")


# ── Section 8 / Phase 3: synthesize Agent 1-3 outputs into JSON audit action ─
def synthesize_audit_action(agent1_result: dict, agent2_result: dict, agent3_result: dict) -> dict:
    """
    agent1_result: {"predicted_cost": float, "model_used": str}
    agent2_result: {"delay_probability": float, "model_used": str}
    agent3_result: {"non_compliance_probability": float, "model_used": str}
    Returns a structured dict (JSON-serialisable) with an executive summary
    and an audit action, using the LLM when available, else a rule-based
    synthesis so the app never breaks without a GPU.
    """
    _try_load()
    if _model is not None:
        prompt = (
            "Given these freight AI agent outputs:\n"
            f"- Predicted shipping cost: ${agent1_result.get('predicted_cost', 0):,.2f}\n"
            f"- Route delay probability: {agent2_result.get('delay_probability', 0)*100:.1f}%\n"
            f"- Carrier non-compliance probability: {agent3_result.get('non_compliance_probability', 0)*100:.1f}%\n\n"
            "Respond with ONLY a JSON object (no markdown fences, no preamble) with keys: "
            "risk_level (Low/Medium/High), executive_summary (2 sentences), "
            "recommended_action (1 sentence), flag_for_manual_review (true/false)."
        )
        try:
            raw = _generate(prompt, max_new_tokens=220)
            match = re.search(r"\{.*\}", raw, re.DOTALL)
            if match:
                parsed = json.loads(match.group(0))
                parsed["_source"] = "llm"
                return parsed
        except Exception as e:
            print(f"[llm_engine_freight] JSON synthesis failed ({e}); using rule-based fallback.")

    return _rule_based_audit_action(agent1_result, agent2_result, agent3_result)


def _rule_based_audit_action(agent1_result, agent2_result, agent3_result) -> dict:
    delay_p = agent2_result.get("delay_probability", 0)
    noncompliance_p = agent3_result.get("non_compliance_probability", 0)
    risk_score = delay_p * 0.5 + noncompliance_p * 0.5
    if risk_score >= 0.6:
        level = "High"
    elif risk_score >= 0.3:
        level = "Medium"
    else:
        level = "Low"
    return {
        "risk_level": level,
        "executive_summary": (
            f"Predicted freight cost is ${agent1_result.get('predicted_cost', 0):,.2f} with a "
            f"{delay_p*100:.1f}% route delay risk and {noncompliance_p*100:.1f}% carrier "
            f"non-compliance risk. Overall shipment risk is assessed as {level}."
        ),
        "recommended_action": (
            "Flag shipment for manual audit before dispatch." if level == "High"
            else "Proceed with standard monitoring." if level == "Low"
            else "Proceed with additional carrier verification."
        ),
        "flag_for_manual_review": level == "High",
        "_source": "rule_based_fallback",
    }


def unload_model():
    """Frees GPU memory (Step 8 — Stop Application)."""
    global _model, _tokenizer, _load_failed
    try:
        import torch, gc
        _model = None
        _tokenizer = None
        _load_failed = False
        gc.collect()
        torch.cuda.empty_cache()
    except Exception:
        pass


In [ ]:
%%writefile app.py
"""
app.py — FreightQuote AI Milestone 2
Main Streamlit application. Gates all tabs behind auth.render_auth_portal(),
then exposes: Home (KPI overview), Agent 1-3 tabs, AI Copilot, and an
Admin Dashboard tab restricted to role == 'Admin'.
"""
import os
import joblib
import pandas as pd
import textwrap
import streamlit as st

from config import (AGENT1_MODEL_PATH, AGENT2_MODEL_PATH, AGENT3_MODEL_PATH,
                     MODELS_DIR, INDIAN_PORTS)
from db import init_db, get_conn, log_chat
from ui_theme import (inject_css, render_card, render_hero, render_page_title,
                       kpi_tile, badge, chat_bubble, COLORS)
from auth import render_auth_portal, verify_jwt
import llm_engine_freight as llm

st.set_page_config(page_title="FreightQuote AI", page_icon="🚛", layout="wide")
inject_css()
init_db()

ss = st.session_state

# ── Auth gate ─────────────────────────────────────────────────────────────────
if not ss.get("token") or not verify_jwt(ss.get("token")):
    render_auth_portal()
    st.stop()

username = ss.get("username", "User")
role = ss.get("role", "User")
is_admin = role.lower() == "admin"

# ── Sidebar ───────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown(textwrap.dedent(f"""
    <div style="text-align:center;padding:6px 0 14px;">
        <div style="font-size:34px;">🚛</div>
        <div style="font-family:'Sora',sans-serif;font-weight:800;font-size:17px;
                    color:{COLORS['text_heading']};margin-top:2px;">FreightQuote AI</div>
        <div style="font-size:11px;color:{COLORS['text_muted']};letter-spacing:0.08em;
                    text-transform:uppercase;">Enterprise Logistics Suite</div>
    </div>
    """), unsafe_allow_html=True)

    st.markdown(textwrap.dedent(f"""
    <div style="background:{COLORS['card_bg']};border:1px solid {COLORS['border']};
                border-radius:12px;padding:10px 14px;margin-bottom:14px;
                display:flex;align-items:center;gap:10px;">
        <div style="width:34px;height:34px;border-radius:50%;
                    background:linear-gradient(135deg,{COLORS['primary']},{COLORS['primary_light']});
                    display:flex;align-items:center;justify-content:center;
                    font-weight:800;color:#04120f;">{username[:1].upper()}</div>
        <div>
            <div style="font-weight:700;color:{COLORS['text_heading']};font-size:14px;">{username}</div>
            <div style="font-size:11px;color:{COLORS['primary_light']};">{role}</div>
        </div>
    </div>
    """), unsafe_allow_html=True)

    if st.button("🚪 Log Out"):
        ss.clear()
        st.rerun()
    st.markdown("---")
    pages = ["🏠 Home", "💰 Agent 1: Pricing", "🚦 Agent 2: Route Delay",
             "🛡️ Agent 3: Carrier Compliance", "🤖 AI Copilot"]
    if is_admin:
        pages.append("⚙️ Admin Dashboard")
    page = st.radio("Navigate", pages, label_visibility="collapsed")


# ── Helpers to load trained models (graceful if not yet trained) ────────────
def _load(path):
    return joblib.load(path) if os.path.exists(path) else None


agent1_model = _load(AGENT1_MODEL_PATH)
agent2_model = _load(AGENT2_MODEL_PATH)
agent3_model = _load(AGENT3_MODEL_PATH)
agent2_scaler = _load(os.path.join(MODELS_DIR, "agent2_scaler.joblib"))
agent3_scaler = _load(os.path.join(MODELS_DIR, "agent3_scaler.joblib"))


# ══════════════════════════════════════════════════════════════════════════
# HOME
# ══════════════════════════════════════════════════════════════════════════
if page == "🏠 Home":
    render_hero("FreightQuote AI", "Executive Overview — Multi-Agent Logistics & Pricing Intelligence", "🏠")

    with get_conn() as conn:
        total_users = conn.execute("SELECT COUNT(*) FROM users").fetchone()[0]
        total_queries = conn.execute(
            "SELECT COUNT(*) FROM chat_history WHERE role='user'"
        ).fetchone()[0]
        champions = pd.read_sql(
            "SELECT agent_name, model_name, metric_primary FROM ml_models WHERE is_champion=1", conn
        )

    k1, k2, k3, k4 = st.columns(4)
    with k1: kpi_tile("Registered Users", total_users, "👥")
    with k2: kpi_tile("Copilot Queries Logged", total_queries, "💬")
    with k3: kpi_tile("Agents Trained", champions["agent_name"].nunique() if not champions.empty else 0, "🧠")
    with k4: kpi_tile("Active Ports Covered", len(INDIAN_PORTS), "🇮🇳")

    st.markdown("<br>", unsafe_allow_html=True)
    render_page_title("Indian Port Coverage", "🇮🇳")
    st.dataframe(pd.DataFrame(INDIAN_PORTS), use_container_width=True, hide_index=True)

    st.markdown("<br>", unsafe_allow_html=True)
    if not champions.empty:
        render_page_title("Current Champion Models", "🏆")
        st.dataframe(champions, use_container_width=True, hide_index=True)
    else:
        st.info("No trained models yet — run the ML training cell (Step 6) in the notebook.")


# ══════════════════════════════════════════════════════════════════════════
# AGENT 1 — PRICING CALCULATOR
# ══════════════════════════════════════════════════════════════════════════
elif page == "💰 Agent 1: Pricing":
    render_page_title("Agent 1 — Dynamic Pricing & Freight Cost Analyzer", "💰")
    if agent1_model is None:
        st.warning("Model not trained yet. Run the ML training cell in the notebook first.")
    else:
        c1, c2, c3 = st.columns(3)
        distance = c1.number_input("Distance (km)", 100, 20000, 4500)
        weight = c2.number_input("Weight (kg)", 1, 50000, 1200)
        fuel = c3.number_input("Fuel Index (0.9 - 1.4)", 0.90, 1.40, 1.10)
        c4, c5, c6 = st.columns(3)
        congestion = c4.selectbox("Port Congestion", [0, 1, 2], format_func=lambda x: ["Low", "Medium", "High"][x])
        cargo_type = c5.selectbox("Cargo Type", [0, 1, 2, 3],
                                   format_func=lambda x: ["General", "Perishable", "Hazmat", "Bulk"][x])
        port_dwell = c6.number_input("Port Dwell Time (days)", 0.0, 15.0, 2.5)

        if st.button("📊 Predict Freight Cost"):
            X = pd.DataFrame([{
                "distance": distance, "weight": weight, "congestion": congestion,
                "fuel": fuel, "cargo_type": cargo_type, "port_dwell": port_dwell,
            }])
            pred = float(agent1_model.predict(X)[0])
            ss["agent1_result"] = {"predicted_cost": pred, "model_used": type(agent1_model).__name__}
            render_card(
                f"<div style='font-size:13px;color:{COLORS['text_muted']};font-weight:600;"
                f"text-transform:uppercase;letter-spacing:0.06em;'>Predicted Freight Cost</div>"
                f"<div style='font-family:Sora,sans-serif;font-size:32px;font-weight:800;"
                f"color:{COLORS['text_heading']};margin-top:4px;'>${pred:,.2f}</div>"
                f"<div style='margin-top:8px;'>{badge('Model: ' + type(agent1_model).__name__, 'accent')}</div>",
                accent="gold",
            )


# ══════════════════════════════════════════════════════════════════════════
# AGENT 2 — ROUTE DELAY CLASSIFIER
# ══════════════════════════════════════════════════════════════════════════
elif page == "🚦 Agent 2: Route Delay":
    render_page_title("Agent 2 — Route Delay & Risk Classifier", "🚦")
    if agent2_model is None or agent2_scaler is None:
        st.warning("Model not trained yet. Run the ML training cell in the notebook first.")
    else:
        c1, c2, c3 = st.columns(3)
        dwell = c1.number_input("Port Dwell (days)", 0.0, 12.0, 3.0)
        berth = c2.number_input("Berth Wait (hrs)", 0, 80, 20)
        route_length = c3.number_input("Route Length (km)", 100, 20000, 5000)
        c4, c5, c6 = st.columns(3)
        weather = c4.slider("Weather Severity", 0.0, 1.0, 0.3)
        canal = c5.selectbox("Passes Canal Chokepoint?", [0, 1], format_func=lambda x: "No" if x == 0 else "Yes")
        season_risk = c6.slider("Seasonal Risk", 0.0, 1.0, 0.3)

        if st.button("📊 Predict Delay Risk"):
            X = pd.DataFrame([{
                "dwell": dwell, "berth": berth, "route_length": route_length,
                "weather": weather, "canal": canal, "season_risk": season_risk,
            }])
            X_scaled = agent2_scaler.transform(X)
            proba = float(agent2_model.predict_proba(X_scaled)[0][1])
            ss["agent2_result"] = {"delay_probability": proba, "model_used": type(agent2_model).__name__}
            kind = "danger" if proba >= 0.6 else "accent" if proba >= 0.3 else "success"
            accent = "danger" if proba >= 0.6 else "gold" if proba >= 0.3 else "success"
            level = "High Risk" if proba >= 0.6 else "Medium Risk" if proba >= 0.3 else "Low Risk"
            render_card(
                f"<div style='font-size:13px;color:{COLORS['text_muted']};font-weight:600;"
                f"text-transform:uppercase;letter-spacing:0.06em;'>Route Delay Risk</div>"
                f"<div style='font-family:Sora,sans-serif;font-size:32px;font-weight:800;"
                f"color:{COLORS['text_heading']};margin-top:4px;'>{proba*100:.1f}%</div>"
                f"<div style='margin-top:8px;'>{badge(level, kind)}</div>",
                accent=accent,
            )


# ══════════════════════════════════════════════════════════════════════════
# AGENT 3 — CARRIER COMPLIANCE SENTINEL
# ══════════════════════════════════════════════════════════════════════════
elif page == "🛡️ Agent 3: Carrier Compliance":
    render_page_title("Agent 3 — Carrier Compliance Sentinel", "🛡️")
    if agent3_model is None or agent3_scaler is None:
        st.warning("Model not trained yet. Run the ML training cell in the notebook first.")
    else:
        c1, c2, c3 = st.columns(3)
        on_time = c1.slider("On-Time Delivery Rate", 0.0, 1.0, 0.9)
        damage = c2.slider("Damage Rate", 0.0, 0.3, 0.03)
        doc_acc = c3.slider("Documentation Accuracy", 0.0, 1.0, 0.95)
        c4, c5, c6 = st.columns(3)
        insurance = c4.selectbox("Insurance Valid?", [0, 1], format_func=lambda x: "No" if x == 0 else "Yes")
        violations = c5.number_input("Prior Violations", 0, 10, 0)
        inspection = c6.number_input("Inspection Score", 0, 100, 85)

        if st.button("📊 Predict Compliance Risk"):
            X = pd.DataFrame([{
                "on_time_rate": on_time, "damage_rate": damage, "doc_accuracy": doc_acc,
                "insurance_valid": insurance, "prior_violations": violations,
                "inspection_score": inspection,
            }])
            X_scaled = agent3_scaler.transform(X)
            proba = float(agent3_model.predict_proba(X_scaled)[0][1])
            ss["agent3_result"] = {"non_compliance_probability": proba, "model_used": type(agent3_model).__name__}
            kind = "danger" if proba >= 0.6 else "accent" if proba >= 0.3 else "success"
            accent = "danger" if proba >= 0.6 else "gold" if proba >= 0.3 else "success"
            level = "High Risk" if proba >= 0.6 else "Watch" if proba >= 0.3 else "Compliant"
            render_card(
                f"<div style='font-size:13px;color:{COLORS['text_muted']};font-weight:600;"
                f"text-transform:uppercase;letter-spacing:0.06em;'>Non-Compliance Probability</div>"
                f"<div style='font-family:Sora,sans-serif;font-size:32px;font-weight:800;"
                f"color:{COLORS['text_heading']};margin-top:4px;'>{proba*100:.1f}%</div>"
                f"<div style='margin-top:8px;'>{badge(level, kind)}</div>",
                accent=accent,
            )


# ══════════════════════════════════════════════════════════════════════════
# AI COPILOT
# ══════════════════════════════════════════════════════════════════════════
elif page == "🤖 AI Copilot":
    render_page_title("AI Copilot — Executive Shipping Strategy", "🤖")
    st.caption("Powered by Qwen2.5-3B-Instruct (4-bit) — falls back to rule-based responses without a GPU.")

    prompt = st.text_area("Ask the Copilot anything about this shipment strategy:",
                           placeholder="Explain in 2 sentences why port congestion increases freight risk.")
    if st.button("💬 Ask Copilot"):
        if prompt.strip():
            with st.spinner("Thinking..."):
                response = llm.copilot_chat(prompt)
            log_chat(username, "user", prompt)
            log_chat(username, "assistant", response)
            chat_bubble("Copilot", response)

    st.markdown("---")
    render_page_title("Structured Audit Action (Phase 3)", "📋")
    st.caption("Synthesises the latest Agent 1-3 predictions above into a JSON audit action.")
    if st.button("🧠 Synthesize Audit Action"):
        a1 = ss.get("agent1_result", {"predicted_cost": 0})
        a2 = ss.get("agent2_result", {"delay_probability": 0})
        a3 = ss.get("agent3_result", {"non_compliance_probability": 0})
        with st.spinner("Synthesizing..."):
            audit = llm.synthesize_audit_action(a1, a2, a3)
        st.json(audit)


# ══════════════════════════════════════════════════════════════════════════
# ADMIN DASHBOARD
# ══════════════════════════════════════════════════════════════════════════
elif page == "⚙️ Admin Dashboard" and is_admin:
    from admin_dash import render_admin_dashboard
    render_admin_dashboard()


## Step 5 — Initialise Database & Seed Admin Account

In [ ]:
import db, auth
db.init_db()
auth.init_auth()
print("✅ Database initialised and admin account seeded.")

## Step 6 — Train ML Agents (5+ Algorithms per Agent)

Trains all 3 agents on their assigned Kaggle datasets (synthetic fallback if
Kaggle credentials aren't set), comparing 7 algorithms each, and saves the
champion model + metrics to the `ml_models` table.

**Confirm Agent 1's printed R² is ≥ 0.90 before moving on.**

In [ ]:
import train_ml_freight
train_ml_freight.train_all_agents()

## Step 7 — Launch Streamlit App via ngrok

Opens a public HTTPS URL. Log in with your `ADMIN_EMAIL_ID` / `ADMIN_PASSWORD`
(or the defaults `infosys@ai` / `admin@123` if you didn't set secrets).

In [ ]:
import subprocess, time, os
from pyngrok import ngrok, conf
from google.colab import userdata

try:
    token = userdata.get("NGROK_AUTHTOKEN")
    if token:
        conf.get_default().auth_token = token
except Exception:
    pass

# ── Forward Colab Secrets into the environment ──────────────────────────────
# app.py runs as its own OS subprocess (via subprocess.Popen below), and
# Colab's userdata.get() only works inside the notebook's own kernel process.
# A separate subprocess cannot see the secrets vault directly, so without this
# step config.py's _secret() call falls through to os.environ (empty) even
# when the secret is set correctly in the sidebar. We read each secret here,
# in the kernel where userdata.get() *does* work, and inject it into
# os.environ — subprocess.Popen inherits the parent's environment by default,
# so app.py's config.py will now find it via the os.environ.get() fallback.
_SECRET_NAMES = [
    "JWT_SECRET_KEY", "ADMIN_EMAIL_ID", "ADMIN_PASSWORD", "NGROK_AUTHTOKEN",
    "HF_TOKEN", "EMAIL_ID", "EMAIL_PASSWORD", "KAGGLE_USERNAME", "KAGGLE_KEY",
]
_forwarded = []
for _name in _SECRET_NAMES:
    try:
        _val = userdata.get(_name)
    except Exception:
        _val = None
    if _val:
        os.environ[_name] = _val
        _forwarded.append(_name)

print(f"🔐 Forwarded {len(_forwarded)} secret(s) into the app environment: "
      f"{', '.join(_forwarded) if _forwarded else '(none found)'}")
if "EMAIL_ID" not in _forwarded or "EMAIL_PASSWORD" not in _forwarded:
    print("⚠️ EMAIL_ID / EMAIL_PASSWORD not both set — OTP emails will use the console fallback.")

ngrok.kill()  # clear any stale tunnels
proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    env=os.environ.copy(),
)
time.sleep(8)
public_url = ngrok.connect(8501, "http")
print(f"🌐 Your app is live at: {public_url}")
print("Login with your ADMIN_EMAIL_ID / ADMIN_PASSWORD secrets (or infosys@ai / admin@123 by default).")


### Verify before capturing screenshots (Section 10.1 / 11)
- [ ] Login works with ADMIN_EMAIL_ID / ADMIN_PASSWORD
- [ ] Home page shows the KPI overview
- [ ] AI Copilot page returns a response
- [ ] ML Pricing Calculator returns a predicted cost
- [ ] Admin Panel → ML Model Card tab shows R²/RMSE and ROC-AUC metrics
- [ ] Progressive lockout, OTP cooldown, and password strength badges behave as specified


## Step 8 — Stop Application & Free GPU Memory

In [ ]:
try:
    proc.terminate()
    print("✅ Streamlit process stopped.")
except Exception as e:
    print(f"Streamlit process already stopped or not found: {e}")

try:
    from pyngrok import ngrok
    ngrok.kill()
    print("✅ ngrok tunnel closed.")
except Exception:
    pass

try:
    import llm_engine_freight
    llm_engine_freight.unload_model()
    print("✅ GPU memory freed.")
except Exception:
    pass

---
### Before uploading to your Infosys Repository
1. Restart and re-run this notebook top to bottom to confirm it works cleanly.
2. **Edit → Clear all outputs** so no tokens or ngrok URLs remain visible.
3. Search for any hard-coded email, JWT secret, ngrok token, Kaggle key, or admin
   password you may have pasted directly into a cell — remove it, keeping only
   the Colab-secrets lookups in `config.py`.
4. Download as `.ipynb`, name it **FreightQuote_AI_Milestone2.ipynb**, and upload
   it into the `Milestone2` folder alongside your `README.md` and `screenshots/`.
